# 04 - Qdrant Ingestion (Hue Foods RAG MVP)

Notebook này trình bày Phase 4 và **kiểm tra read-only** candidate collection `hue_foods_e5_small_384_dense` trong Qdrant thật. Run All **không** chạy ingestion, không upsert, không reset, không delete - chỉ đọc metadata, schema và payload projection an toàn.

**Prerequisite**

- Qdrant local đang chạy qua Docker Compose (image pinned v1.18.3, port 6333).
- Candidate collection `hue_foods_e5_small_384_dense` đã được ingestion tạo trước đó với đúng 572 points.

**Kết quả mong đợi khi Run All**

- Collection tồn tại, schema khớp: pure dense 384 cosine (không có sparse vectors).
- Exact count = 572 points.
- Payload mẫu chỉ in các field metadata đã phê duyệt, không in toàn bộ text.

Nếu Qdrant tắt, collection thiếu, schema lệch hoặc count khác 572, notebook fail rõ ràng - đó là hành vi mong muốn.


In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
else:
    raise RuntimeError(
        "Khong tim thay thu muc backend/. Hay mo notebook nay tu repo root "
        "hoac tu thu muc notebooks/."
    )
print(f"backend on path: {sys.path[0]}")


## Cấu hình vector database (`settings.yaml`)

Nhóm `vector_database` khai báo Qdrant local: URL, dimension 384, cosine, timeout 30 giây.


In [ ]:
from core.settings_loader import load_settings

settings = load_settings()
db = settings["vector_database"]
print("url:", db["url"])
print("configured active collection:", db["collection_name"])
print("vector_size:", db["vector_size"])
print("distance:", db["distance"])
print("timeout:", db["timeout"])


## Point contract và UUID5 deterministic

Mỗi point ID là `uuid.uuid5(uuid.NAMESPACE_URL, f"hue-rag:{chunk_id}")`. Cùng một `chunk_id` luôn ra cùng một ID nên upsert là idempotent; original `chunk_id` được giữ trong payload.


In [ ]:
from vectorstore.points import point_id_for

chunk_id = "foods/restaurants/example.md|Tóm tắt|0"
first = point_id_for(chunk_id)
print("point id:", first)
print("deterministic:", point_id_for(chunk_id) == first)
print("different chunk:", point_id_for("foods/cafes/other.md|Tóm tắt|0") != first)


## Schema kỳ vọng của collection

`expected_schema` mô tả named vectors: `dense` (size 384, cosine). Runtime chỉ tạo collection khi absent; collection đã tồn tại phải khớp schema nếu không pipeline fail closed.


In [ ]:
from vectorstore.qdrant import expected_schema

schema = expected_schema(settings)
print("vector names:", sorted(schema))
print("dense size:", schema["dense"].size)
print("dense distance:", schema["dense"].distance)


## Kiểm tra read-only candidate collection thật

Cell dưới kết nối Qdrant thật và chỉ thực hiện các thao tác read-only trên candidate collection `hue_foods_e5_small_384_dense`: `collection_exists`, `get_collection`, `validate_collection_info` (strict dense-only), `count` exact và `scroll` hai payload với `with_vectors=False`. Payload chỉ in các field metadata đã phê duyệt và độ dài text - không in toàn bộ nội dung chunk.


In [ ]:
from vectorstore.qdrant import client_from_settings, validate_collection_info

client = client_from_settings(settings)
candidate_name = "hue_foods_e5_small_384_dense"

if not client.collection_exists(candidate_name):
    raise RuntimeError(
        f"Collection {candidate_name} khong ton tai. Hay chay candidate ingestion "
        "truoc khi chay notebook nay."
    )

info = client.get_collection(candidate_name)
validate_collection_info(info, settings, strict_dense_only=True)

params = info.config.params
count = client.count(candidate_name, exact=True).count
if count != 572:
    raise RuntimeError(f"Expected 572 points, found {count}")

print("collection:", candidate_name)
print("status:", info.status)
print("dense:", params.vectors["dense"].size, params.vectors["dense"].distance)
print("sparse_vectors:", params.sparse_vectors)
print("point count (exact):", count)

records, _ = client.scroll(candidate_name, limit=2, with_payload=True, with_vectors=False)
APPROVED_FIELDS = (
    "chunk_id", "source", "title", "section", "category",
    "subcategory", "chunk_type", "embedding_model",
)
for record in records:
    payload = record.payload or {}
    print({key: payload.get(key) for key in APPROVED_FIELDS})
    print("  text length:", len(payload.get("text", "")))


## Checklist xác nhận Phase 4

1. Config hiển thị đầy đủ thông số vector database.
2. `point_id_for` deterministic cho cùng chunk_id.
3. Schema kỳ vọng đúng dense 384 cosine (pure dense).
4. Candidate collection tồn tại, schema khớp strict dense-only, status green, exact count = 572.
5. Payload mẫu chỉ in approved metadata fields (không có `embedding_dimension`) + độ dài text.
6. Không có thao tác mutation nào trong notebook này.
